# Training Hub Callback Smoke — RHOAIENG-77626

Small **real** Unsloth e2e check of the **unified** 9-hook API (not every HF-only hook).

Exercises: `on_train_begin/end`, `on_epoch_begin/end`, `on_step_begin/end`, `on_log`, `on_save`, `on_evaluate`.

**Not** required by RHOAIENG-54744: HF-only hooks like `on_pre_optimizer_step`, `on_predict`.

## 1. Setup

Clear workbench MLflow env, install the local checkout.

In [ ]:
import os
import subprocess
import sys

CHECKOUT = "/opt/app-root/src/training_hub_checkout"

for key in (
    "MLFLOW_TRACKING_URI",
    "MLFLOW_EXPERIMENT_NAME",
    "MLFLOW_RUN_NAME",
    "MLFLOW_TRACKING_AUTH",
):
    os.environ.pop(key, None)

os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = "0.0.0+callbacksmoke"

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", CHECKOUT, "--no-deps"],
)
print("Installed:", CHECKOUT)

In [ ]:
import training_hub
from training_hub import TrainingHubCallback, TrainingHubContext, lora_sft

print("training_hub:", training_hub.__file__)
assert "training_hub_checkout" in training_hub.__file__, (
    "Not using the patched checkout — re-run the setup cell"
)
print("Imports OK")

## 2. Tiny train + eval data + callback

In [ ]:
import json
from pathlib import Path

OUT = Path("/opt/app-root/src/callback_smoke_out")
DATA = Path("/opt/app-root/src/callback_smoke_data.jsonl")
EVAL = Path("/opt/app-root/src/callback_smoke_eval.jsonl")
OUT.mkdir(parents=True, exist_ok=True)

train_rows = [
    {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
    {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
    {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
    {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
]
eval_rows = train_rows[:2]

for path, rows in ((DATA, train_rows), (EVAL, eval_rows)):
    with path.open("w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

print(f"train={DATA} ({len(train_rows)}) eval={EVAL} ({len(eval_rows)})")


class SmokeLogger(TrainingHubCallback):
    def __init__(self):
        self.events: list[str] = []

    def _record(self, name: str, msg: str) -> None:
        self.events.append(name)
        print(msg, flush=True)

    def on_train_begin(self, context: TrainingHubContext) -> None:
        self._record("on_train_begin", f"[BEGIN] output_dir={context.output_dir}")

    def on_epoch_begin(self, context: TrainingHubContext) -> None:
        self._record("on_epoch_begin", f"[EPOCH_BEGIN] epoch={context.epoch}")

    def on_step_begin(self, context: TrainingHubContext) -> None:
        self._record("on_step_begin", f"[STEP_BEGIN] step={context.step}")

    def on_log(self, context: TrainingHubContext) -> None:
        self._record(
            "on_log",
            f"[LOG] step={context.step} loss={context.loss} lr={context.learning_rate}",
        )

    def on_evaluate(self, context: TrainingHubContext) -> None:
        self._record("on_evaluate", f"[EVAL] step={context.step} metrics={context.metrics}")

    def on_save(self, context: TrainingHubContext) -> None:
        self._record("on_save", f"[SAVE] step={context.step} output_dir={context.output_dir}")

    def on_step_end(self, context: TrainingHubContext) -> None:
        self._record("on_step_end", f"[STEP_END] step={context.step} loss={context.loss}")

    def on_epoch_end(self, context: TrainingHubContext) -> None:
        self._record("on_epoch_end", f"[EPOCH_END] epoch={context.epoch}")

    def on_train_end(self, context: TrainingHubContext) -> None:
        self._record("on_train_end", f"[END] step={context.step}")


callback = SmokeLogger()
print("Callback ready")

## 3. Run tiny LoRA SFT

`save_steps=2` and `eval_steps=2` so `on_save` / `on_evaluate` fire during 4 steps.

In [ ]:
print("Starting lora_sft smoke...", flush=True)

result = lora_sft(
    model_path="Qwen/Qwen2.5-0.5B-Instruct",
    data_path=str(DATA),
    ckpt_output_dir=str(OUT),
    num_epochs=1,
    max_seq_len=128,
    micro_batch_size=1,
    logging_steps=1,
    save_steps=2,
    eval_steps=2,
    eval_data_path=str(EVAL),
    save_total_limit=2,
    warmup_steps=0,
    learning_rate=2e-4,
    lora_r=8,
    lora_alpha=16,
    sample_packing=False,
    callbacks=[callback],
)

print("Training finished")

## 4. Pass / fail — all 9 unified hooks

In [ ]:
from collections import Counter

counts = Counter(callback.events)
print("Hook counts:", dict(counts))

required = [
    "on_train_begin",
    "on_epoch_begin",
    "on_step_begin",
    "on_log",
    "on_evaluate",
    "on_save",
    "on_step_end",
    "on_epoch_end",
    "on_train_end",
]
missing = [h for h in required if counts[h] < 1]

if missing:
    raise AssertionError(f"SMOKE FAILED — missing hooks: {missing}")

print("SMOKE OK — all 9 unified hooks fired on Unsloth")
print(f"Checkpoint dir: {OUT}")